### Module 8: Conclusion

Everything above builds toward one place: answering the four questions from Module 0, as directly
and traceably as possible. This module summarizes what we did, states every major finding together
with the single number that decided it, ranks those deciding factors against each other, and closes
with the project's bottom-line conclusion.


#### Summary

**What we analysed.** 52,704 hourly air-quality readings across six cities (Brasilia, Cairo,
Dubai, London, New York, Sydney) for all of 2024, covering CO, CO2, NO2, SO2, O3, PM2.5, PM10, and
AQI, in order to answer the four questions posed in Module 0.

**What the work actually involved.** We built this up in the same order the course taught it:
plain variables and operators first (Aishmita), then loops and functions applied to a small
hand-typed sample before touching any library, then real file reading with `open()`, and only
after all of that did we bring in Pandas for the full 52,704-row dataset (Aaska). NumPy and
Matplotlib came last, once there was a clean DataFrame worth visualizing (Anupam).

**What we found.** Dubai is consistently the most polluted city and Brasilia the cleanest. AQI
moves in a fairly narrow band across the year but spikes hard in the Northern Hemisphere summer,
and almost all of the dataset's worst hours trace back to a single mid-August event in Dubai
rather than a general summer trend. Particulate matter (PM2.5 and PM10) is clearly linked to AQI,
which makes sense given how AQI is built.


#### Major findings

Every number below comes from the outputs above, and we checked each one against the printed
result before writing it here. For each finding we also name the **deciding factor** -- the
specific metric or threshold that is doing the actual work of supporting the conclusion -- so the
reasoning is traceable back to a single number, not just a general impression.

**1. Dubai has by far the worst air quality, Brasilia the best.**
Dubai averages an AQI of 85.1 across the whole year, well into "Moderate" territory and close to
"Unhealthy for Sensitive Groups." Brasilia averages 24.7, the cleanest of the six, with Sydney
(25.8) and London (27.1) close behind it; Cairo (52.5) and New York (32.9) fall in between.
**Deciding factor:** the `city_summary` groupby-mean of `AQI` (Module 6). A roughly 60-point gap
between the highest and lowest city average is far larger than normal year-to-year variation for
any single city, which is why city -- not season or random noise -- is the dominant factor here.

**2. AQI stays fairly steady across the year, with July and August the worst months.**
The monthly average never leaves the 36-48 range, but it clearly peaks in July (46.8) and August
(48.3) and dips lowest in October (35.9) -- a swing of about 12 points, roughly a fifth of Dubai's
total average.
**Deciding factor:** the `monthly_aqi` groupby-mean of `AQI` (Module 6). The swing is real but
small relative to the *between-city* gap in Finding 1, which is why we rank city above month as a
driver of AQI.

**3. One event drives almost all of the "Unhealthy" hours in the dataset.**
2,182 of 52,704 hourly readings (4.1%) exceed AQI 100, but every one of the ten worst hours in the
entire dataset happened in Dubai on 18 August 2024.
**Deciding factor:** sorting `high_aqi_df` by AQI descending (Module 6) and reading the `City` and
`Date` of the top 10 rows -- all ten share the same city *and* the same date, which rules out a
general seasonal effect and points to a single short-lived pollution event instead.

**4. Particulate matter is strongly linked to AQI.**
PM2.5 correlates with AQI at r = 0.82 and PM10 at r = 0.85.
**Deciding factor:** `np.corrcoef()` on the raw hourly arrays (Module 6). Both coefficients clear
the common "strong correlation" threshold of |r| > 0.7, which is why we call particulate matter the
dominant pollutant input to AQI in this dataset -- consistent with AQI being partly calculated from
particulate readings in the first place, so this confirms the relationship rather than discovering
a new one.

**5. The missing CO2 data is a monitoring gap, not random noise.**
81.7% of CO2 readings are missing, but not randomly: CO2 monitoring only started on 26 October
2024, identically across all six cities.
**Deciding factor:** the `co2_availability_pattern()` breakdown by month (Module 6) shows zero CO2
readings for every city before late October, then near-complete coverage afterward. A hard cutoff
at a single date, shared across every city, is the signature of a reporting start date rather than
sensor drift or random missingness -- which is why we excluded CO2 from the main year-long city
comparison instead of imputing or dropping it outright.

**6. Even inside that short CO2 window, city rankings looked different.**
New York had the highest average CO2 (about 488 ppm) and Sydney the lowest (about 443 ppm) in the
Oct 26 - Dec 31 window -- not the same order as the full-year AQI ranking (Dubai worst, Brasilia
best).
**Deciding factor:** `summarize_co2_availability()`'s groupby-mean of CO2 by city (Module 6). The
mismatch between the CO2 ranking and the AQI ranking is itself the deciding factor for keeping CO2
out of the main city comparison -- a pollutant that ranks cities differently from the overall index
cannot be used as a stand-in for overall air quality.


#### Conclusion analysis -- what actually decided each conclusion

Each conclusion above rests on one or two specific numbers, not a general impression of the data.
Ranked by how much each factor actually moved the analysis:

| Rank | Deciding factor | Why it outweighed the alternatives |
| :--- | :--- | :--- |
| 1 | **City** (about 60-point AQI gap, Dubai vs. Brasilia) | The single largest source of variation in the dataset -- more than 4x the size of the month-to-month swing (rank 4) -- and the reason the analysis is organized around `groupby("City")` rather than a single dataset-wide average. |
| 2 | **A single event** (Dubai, 18 August 2024) | Explains Finding 3 better than "summer is worse" alone: all ten worst hours in the entire dataset share one city *and* one date, which a seasonal effect by itself would not produce. |
| 3 | **Particulate correlation** (PM10 r = 0.85, PM2.5 r = 0.82) | Both comfortably clear the r > 0.7 "strong correlation" bar, and PM10 edges out PM2.5 -- which is why particulate levels, not CO, NO2, SO2, or O3, are the pollutants we identify as AQI's main driver. |
| 4 | **Seasonality** (Jul/Aug peak vs. Oct trough, a 12-point swing) | Real and consistent, but the smallest of the four factors -- about a fifth of the city-level gap in rank 1 -- so we treat it as a secondary rather than primary driver of AQI. |

This ranking is also why the missing-CO2 decision (Finding 5) was made the way it was: CO2 never
entered the table above, because the data could not support including it honestly across the full
year. We chose to under-use a column rather than let 10 months of invented values quietly bias the
city or seasonal rankings that the rest of this conclusion depends on.


#### Final conclusion

Dubai has the worst air quality of the six cities we studied (AQI 85.1), Brasilia the best (24.7)
-- a gap big enough that **which city you're in matters more than what month it is**. Pollution
peaks in July/August and dips in October, but that seasonal swing is smaller than the gap between
cities. Almost all of the year's worst hours trace back to a single event in Dubai on 18 August
2024, not a general trend. Particulate matter (PM2.5, PM10) is the strongest driver of AQI,
correlating at r = 0.82 and r = 0.85. And on the data itself: 81.7% of CO2 readings are missing
because monitoring only started in late October, so we reported that gap honestly instead of
guessing at ten months of numbers.

**Bottom line:** air quality in this dataset is driven far more by *where* you are than *when* --
and the single worst event we found was a one-off spike in Dubai, not a seasonal pattern.

The full reasoning behind each of these points, including the deciding factor we used to reach it,
is in the Major Findings and Conclusion Analysis sections above.
